<a href="https://colab.research.google.com/github/HazemmoAlsady/AWN_Graduation_Project/blob/main/classification_model2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [2]:
!kaggle datasets download -d kiva/data-science-for-good-kiva-crowdfunding

Dataset URL: https://www.kaggle.com/datasets/kiva/data-science-for-good-kiva-crowdfunding
License(s): CC0-1.0
100% 41.9M/41.9M [00:04<00:00, 10.7MB/s]



In [3]:
!unzip /content/data-science-for-good-kiva-crowdfunding.zip -d dataset

Archive:  /content/data-science-for-good-kiva-crowdfunding.zip
  inflating: dataset/kiva_loans.csv  
  inflating: dataset/kiva_mpi_region_locations.csv  
  inflating: dataset/loan_theme_ids.csv  
  inflating: dataset/loan_themes_by_region.csv  


In [4]:
!ls dataset

kiva_loans.csv		       loan_theme_ids.csv
kiva_mpi_region_locations.csv  loan_themes_by_region.csv


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import xgboost as xgb

In [6]:
import pandas as pd

df = pd.read_csv("/content/dataset/kiva_loans.csv")
df


,id,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,partner_id,posted_time,disbursed_time,funded_time,term_in_months,lender_count,tags,borrower_genders,repayment_interval,date
0,653051,300.0,300.0,Fruits & Vegetables,Food,"To buy seasonal, fresh fruits to sell.",PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:12:39+00:00,2013-12-17 08:00:00+00:00,2014-01-02 10:06:32+00:00,12.0,12,NaN,female,irregular,2014-01-01
1,653053,575.0,575.0,Rickshaw,Transportation,to repair and maintain the auto rickshaw used ...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:51:08+00:00,2013-12-17 08:00:00+00:00,2014-01-02 09:17:23+00:00,11.0,14,NaN,"female, female",irregular,2014-01-01
2,653068,150.0,150.0,Transportation,Transportation,To repair their old cycle-van and buy another ...,IN,India,Maynaguri,INR,334.0,2014-01-01 09:58:07+00:00,2013-12-17 08:00:00+00:00,2014-01-01 16:01:36+00:00,43.0,6,"user_favorite, user_favorite",female,bullet,2014-01-01
3,653063,200.0,200.0,Embroidery,Arts,to purchase an embroidery machine and a variet...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 08:03:11+00:00,2013-12-24 08:00:00+00:00,2014-01-01 13:00:00+00:00,11.0,8,NaN,female,irregular,2014-01-01
4,653084,400.0,400.0,Milk Sales,Food,to purchase one buffalo.,PK,Pakistan,Abdul Hakeem,PKR,245.0,2014-01-01 11:53:19+00:00,2013-12-17 08:00:00+00:00,2014-01-01 19:18:51+00:00,14.0,16,NaN,female,monthly,2014-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
671200,1340323,0.0,25.0,Livestock,Agriculture,"[True, u'para compara: cemento, arenya y ladri...",PY,Paraguay,Concepción,USD,58.0,2017-07-25 16:55:34+00:00,2017-07-25 07:00:00+00:00,NaN,13.0,0,NaN,female,monthly,2017-07-25
671201,1340316,25.0,25.0,Livestock,Agriculture,"[True, u'to start a turducken farm.'] - this l...",KE,Kenya,NaN,KES,138.0,2017-07-25 06:14:08+00:00,2017-07-24 07:00:00+00:00,2017-07-26 02:09:43+00:00,13.0,1,NaN,female,monthly,2017-07-25
671202,1340334,0.0,25.0,Games,Entertainment,NaN,KE,Kenya,NaN,KES,138.0,2017-07-26 00:02:07+00:00,2017-07-25 07:00:00+00:00,NaN,13.0,0,NaN,NaN,monthly,2017-07-26
671203,1340338,0.0,25.0,Livestock,Agriculture,"[True, u'to start a turducken farm.'] - this l...",KE,Kenya,NaN,KES,138.0,2017-07-26 06:12:55+00:00,2017-07-25 07:00:00+00:00,NaN,13.0,0,NaN,female,monthly,2017-07-26


In [7]:
df.isnull().sum()

,0
id,0
funded_amount,0
loan_amount,0
activity,0
sector,0
use,4232
country_code,8
country,0
region,56800
currency,0


In [8]:
df = df[["use", "sector"]]

df.dropna(inplace=True)

df.reset_index(drop=True, inplace=True)


/tmp/ipykernel_6237/1557571249.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


In [9]:
import re
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

df["request_text"] = df["use"].apply(clean_text)

/tmp/ipykernel_6237/3021933747.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["request_text"] = df["use"].apply(clean_text)


In [10]:
def map_label(sector):
    if sector in ["Agriculture", "Retail", "Transportation", "Services",
                  "Manufacturing", "Clothing", "Construction", "Wholesale"]:
        return "مالي"
    elif sector == "Education":
        return "تعليمي"
    elif sector == "Health":
        return "صحي"
    elif sector == "Housing":
        return "سكني"
    elif sector == "Food":
        return "غذاء"
    else:
        return "مالي"

df["label"] = df["sector"].apply(map_label)
df = df[["request_text", "label"]]

print("توزيع الفئات قبل التوازن:")
print(df["label"].value_counts())

توزيع الفئات قبل التوازن:
label
مالي      457646
غذاء      135747
سكني       33571
تعليمي     30837
صحي         9172
Name: count, dtype: int64


/tmp/ipykernel_6237/225147026.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["label"] = df["sector"].apply(map_label)


In [11]:
min_count = df["label"].value_counts().min()
df_balanced = df.groupby("label").apply(lambda x: x.sample(min_count, random_state=42)).reset_index(drop=True)

print("\nتوزيع الفئات بعد التوازن:")
print(df_balanced["label"].value_counts())


توزيع الفئات بعد التوازن:
label
تعليمي    9172
سكني      9172
صحي       9172
غذاء      9172
مالي      9172
Name: count, dtype: int64


/tmp/ipykernel_6237/98349364.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby("label").apply(lambda x: x.sample(min_count, random_state=42)).reset_index(drop=True)


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df_balanced["request_text"])


In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df_balanced["label"])


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [15]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [16]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred ))

              precision    recall  f1-score   support

           0       0.98      0.97      0.98      1835
           1       0.95      0.98      0.96      1834
           2       0.98      0.93      0.95      1834
           3       0.87      0.88      0.88      1835
           4       0.82      0.84      0.83      1834

    accuracy                           0.92      9172
   macro avg       0.92      0.92      0.92      9172
weighted avg       0.92      0.92      0.92      9172



In [17]:
test_text = ["To buy seasonal, fresh fruits to sell."]


test_vec = vectorizer.transform(test_text)
prediction = model.predict(test_vec)
print("Prediction:", le.inverse_transform(prediction))

Prediction: ['غذاء']


In [19]:
test_text = ["I need money to buy books for school"]

test_vec = vectorizer.transform(test_text)
prediction = model.predict(test_vec)

print(le.inverse_transform(prediction))

['تعليمي']


In [18]:
tests = [
    "I want to buy food for my family",
    "Need money for school fees",
    "Medical treatment for my child",
    "Build a small house",
    "Start a small business"
]

test_vec = vectorizer.transform(tests)
preds = model.predict(test_vec)

for text, pred in zip(tests, preds):
    print(text, "→", le.inverse_transform([pred])[0])

I want to buy food for my family → غذاء
Need money for school fees → تعليمي
Medical treatment for my child → صحي
Build a small house → سكني
Start a small business → غذاء


In [21]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.1 MB/s eta 0:00:00


In [22]:
from deep_translator import GoogleTranslator

def predict_arabic(text_ar):
    # 1️⃣ ترجمة عربي → إنجليزي
    text_en = GoogleTranslator(source='auto', target='en').translate(text_ar)

    # 2️⃣ تحويل لـ vector
    vec = vectorizer.transform([text_en])

    # 3️⃣ prediction
    pred = model.predict(vec)

    # 4️⃣ رجوع label عربي
    result = le.inverse_transform(pred)[0]

    return result, text_en

In [23]:
arabic_tests = [
    "عايز فلوس أبدأ مشروع صغير",
    "محتاج أشتري أكل لأسرتي",
    "عايز أدفع مصاريف المدرسة",
    "محتاج علاج لابني",
    "عايز أبني بيت صغير",
    "عايز أشتري بضاعة للمحل",
    "محتاج أجيب خضار وفاكهة للبيع",
    "عايز أوسع المشروع بتاعي",
    "محتاج أصلح البيت",
    "عايز أشتري أدوية",
    "محتاج أبدأ مشروع تربية مواشي",
    "عايز أفتح كشك صغير",
    "محتاج فلوس للجامعة",
    "عايز أجيب أدوات مدرسية",
    "محتاج علاج عملية جراحية",
    "عايز أشتري مواد بناء",
    "محتاج أبدأ مشروع أكل",
    "عايز أشتري ماكينة خياطة",
    "محتاج فلوس لمصاريف البيت",
    "عايز أفتح مطعم صغير"
]

In [24]:
for text in arabic_tests:
    result, translated = predict_arabic(text)
    print(f"AR: {text}")
    print(f"EN: {translated}")
    print(f"Prediction: {result}")
    print("-" * 50)

AR: عايز فلوس أبدأ مشروع صغير
EN: I want money to start a small project
Prediction: مالي
--------------------------------------------------
AR: محتاج أشتري أكل لأسرتي
EN: I need to buy food for my family
Prediction: غذاء
--------------------------------------------------
AR: عايز أدفع مصاريف المدرسة
EN: I want to pay school fees
Prediction: تعليمي
--------------------------------------------------
AR: محتاج علاج لابني
EN: I need treatment for my son
Prediction: صحي
--------------------------------------------------
AR: عايز أبني بيت صغير
EN: I want to build a small house
Prediction: سكني
--------------------------------------------------
AR: عايز أشتري بضاعة للمحل
EN: I want to buy goods for the store
Prediction: غذاء
--------------------------------------------------
AR: محتاج أجيب خضار وفاكهة للبيع
EN: I need to get vegetables and fruits for sale
Prediction: غذاء
--------------------------------------------------
AR: عايز أوسع المشروع بتاعي
EN: I want to expand my project
Prediction: